In [1]:
# 必要なモジュールをインポート
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from openai.types.chat import ChatCompletionToolParam
from tavily import TavilyClient

# 環境変数の取得
load_dotenv("../.env")

# OpenAI APIクライアントを生成
client = OpenAI(api_key=os.environ['API_KEY'])

# tavily検索用APIキーの取得
TAVILY_API_KEY = os.environ['TAVILY_API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini"

In [2]:
# 検索結果を返す関数の作成
def get_search_result(question):
    client = TavilyClient(api_key=TAVILY_API_KEY)
    response = client.search(question)
    return json.dumps({"result": response["results"]})

In [3]:
# テスト用コード
ret = get_search_result("東京駅のイベントを教えて")
json.loads(ret)

{'result': [{'url': 'https://www.walkerplus.com/event_list/ar0313/sc309880d/',
   'title': '東京駅(東京都)周辺のイベント - ウォーカープラス',
   'content': '開催中 2025年12月20日(土)～2026年2月22日(日). 京橋駅(東京都), 宝町駅(東京都), 日本橋駅(東京都), 銀座一丁目駅(東京都), 東京駅(東京都). CREATIVE MUSEUM TOKYO(クリエイティブ ミュージアム トウキョウ). 日比谷駅(東京都), 有楽町駅(東京都), 東京駅(東京都), 京橋駅(東京都). 開催中 2026年1月2日(金)～3月22日(日). 二重橋前駅(東京都), 有楽町駅(東京都), 東京駅(東京都), 日比谷駅(東京都), 銀座一丁目駅(東京都). * 歴史リアル謎解きゲーム「謎の城」in 日本橋「発明家／人斬り-平賀源内-」. 開催中 2025年9月1日(月)～2026年3月1日(日). 日本橋駅(東京都), 京橋駅(東京都), 東京駅(東京都), 宝町駅(東京都), 三越前駅(東京都). * MIDTOWN YAESU CHRISTMAS 2025 (ミッドタウン八重洲クリスマス2025). 終了間近 2025年11月13日(木)～2026年2月15日(日). 京橋駅(東京都), 東京駅(東京都), 宝町駅(東京都), 日本橋駅(東京都), 銀座一丁目駅(東京都). * 丸の内から御食国「敦賀・若狭」へ 美し物-うましもの-グルメフェア. 開催中 2026年2月2日(月)～21日(土). 二重橋前駅(東京都), 有楽町駅(東京都), 東京駅(東京都), 日比谷駅(東京都). 終了間近 2025年11月13日(木)～2026年2月15日(日). 二重橋前〈丸の内〉駅(東京都), 東京駅(東京都), 有楽町駅(東京都), 日比谷駅(東京都), 銀座一丁目駅(東京都). * Masking Tape Jamboree in KITTE2026. 終了間近 2026年2月11日(水)～13日(金). 東京駅(東京都), 二重橋前駅(東京都), 有楽町駅(東京都), 京橋駅(東京都), 銀座一丁目駅(東京都). 東京駅(東京都), 二重橋前駅

In [4]:
# ツール定義
def define_tools():
    print("------define_tools(ツール定義)------")
    return [
        ChatCompletionToolParam({
            "type": "function",
            "function": {
                "name": "get_search_result",
                "description": "最近一ヵ月のイベント開催予定などネット検索が必要な場合に、質問文の検索結果を取得する",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string", "description": "質問文"},
                    },
                    "required": ["question"],
                },
            },
        })
    ]

In [5]:
# 言語モデルへの質問を行う関数
def ask_question(question, tools):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": question}],
        tools=tools,
        tool_choice="auto",
    )
    return response

In [6]:
# ツール呼び出しが必要な場合の処理を行う関数
def handle_tool_call(response, question):
    # 関数の実行と結果取得
    tool = response.choices[0].message.tool_calls[0]
    function_name = tool.function.name
    arguments = json.loads(tool.function.arguments)
    function_response = globals()[function_name](**arguments)

    # 関数の実行結果をmessagesに加えて再度言語モデルを呼出
    response_after_tool_call = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "user", "content": question},
            response.choices[0].message,
            {
                "tool_call_id": tool.id,
                "role": "tool",
                "content": function_response,
            },
        ],
    )
    return response_after_tool_call

In [7]:
# ユーザーからの質問を処理する関数
def process_response(question, tools):
    response = ask_question(question, tools)

    if response.choices[0].finish_reason == 'tool_calls':
        # ツール呼出の場合
        final_response = handle_tool_call(response, question)
        return final_response.choices[0].message.content.strip()
    else:
        # 言語モデルが直接回答する場合
        return response.choices[0].message.content.strip()

In [8]:
tools = define_tools()

# 言語モデルが直接回答できる質問
question = "東京都と沖縄県はどちらが広いですか？"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
東京都と沖縄県の面積を比較すると、沖縄県の方が広いです。具体的な面積は以下の通りです。

- **東京都**: 約2,194平方キロメートル
- **沖縄県**: 約2,271平方キロメートル

したがって、沖縄県の方が東京都よりも広いです。


In [9]:
tools = define_tools()

# ツール呼出が必要な質問
question = "東京駅のイベントについて、最近1ヶ月以内の検索結果を教えてください"
response_message = process_response(question, tools)
print(response_message)

------define_tools(ツール定義)------
最近1ヶ月以内の東京駅でのイベント情報をいくつかご紹介します。

1. **[新商品の展示会](https://www.kotsukaikan.co.jp/business/exhibition/)**  
   東京交通会館で展示会が開催されており、様々な新製品を紹介しています。詳細は、展示会場にて。

2. **[Tokyo City i](https://www.tokyocity-i.jp/)**  
   東京駅近くの観光情報センターでは、近隣の観光スポットやイベント情報を提供しています。

3. **useums around Tokyo Station](https://6museums.tokyo/)**  
   東京駅周辺の美術館を巡るイベントが開催中です。詳細は6つの美術館の協力による情報が掲載されています。

4. **[楽しめるイベント一覧](https://www.enjoytokyo.jp/event/list/)**  
   東京の各地で開催されているイベントやアクティビティの情報を網羅した一覧が閲覧可能です。

これらのイベントは、特に東京駅周辺での文化体験や展示会に焦点を当てています。興味のある方は、各リンクからさらなる情報を確認してください。


In [10]:
# チャットボットへの組み込み
tools = define_tools()

messages=[]

while(True):
    # ユーザーからの質問を受付
    question = input("メッセージを入力:")
    # 質問が入力されなければ終了
    if question.strip()=="":
        break
    display(f"質問:{question}")

    # メッセージにユーザーからの質問を追加
    messages.append({"role": "user", "content": question.strip()})
    # やりとりが8を超えたら古いメッセージから削除
    if len(messages) > 8:
        del_message = messages.pop(0)

    # 言語モデルに質問
    response_message = process_response(question, tools)

    # メッセージに言語モデルからの回答を追加
    print(response_message, flush=True)
    messages.append({"role": "assistant", "content": response_message})

print("\n---ご利用ありがとうございました！---")

------define_tools(ツール定義)------


'質問:こんにちは'

こんにちは！今日はどんなことをお手伝いできますか？


'質問:東北6県は？'

東北地方は日本の地域の一つで、以下の6つの県から成り立っています。

1. 青森県 (あおもりけん)
2. 岩手県 (いわてけん)
3. 宮城県 (みやぎけん)
4. 秋田県 (あきたけん)
5. 山形県 (やまがたけん)
6. 福島県 (ふくしまけん)

これらの県は、自然豊かで文化的な魅力も多い地域です。


'質問:宮城県のお土産について検索した結果を教えて'

宮城県のお土産には、地元の名産品や特産を活かした様々な品があります。以下は、特におすすめの宮城県のお土産です。

1. **牛たん** - 宮城を代表する料理で、多くの専門店があります。特に、炭火焼きの牛たんは人気です。

2. **ずんだ餅** - 枝豆を使った甘い餡が特徴の和菓子で、色鮮やかで美味しいです。

3. **笹かまぼこ** - 新鮮な魚を使用したかまぼこで、風味豊か。小麦粉で作った皮に包まれて焼かれています。

4. **仙台味噌** - 深い味わいが特徴の仙台味噌は、煮物や味噌汁にぴったりです。

5. **一ノ関銘菓「まめぶ」** - 大豆を原料にしたお菓子で、香ばしい味わいが魅力。

6. **お土産にぴったりの「ずんだシェイク」** - 柔らかいずんだと牛乳を混ぜたドリンクで、爽やかな味わいを楽しめます。

7. **鬼寄せまんじゅう** - 地元の素材を使ったおまんじゅうで、餡の種類も豊富です。

8. **みやぎの地酒** - 美味しい地酒も多く、日本酒好きには嬉しいお土産です。

これらの品々は、宮城県内の特定の店舗で購入できます。詳細や個々の商品の特徴については、各店舗の情報を参照してください。

---ご利用ありがとうございました！---
